# MANO fit on the GPU (Kaggle)

Runs `fit_batch.py --fit` on the packed sessions and gives back one `*_params.npz` per session.

### Before running
1. On your PC: `python fit_batch.py --pack sessions/...` writes `fit_inputs/<session>.npz` (~10-15 MB each).
2. Kaggle: **Create, New Dataset**, upload the `.npz` files from `fit_inputs/` (not the `*_params.npz`), any name.
   Keep it private. It holds only point clouds and joints, no MANO files.
3. In this notebook: **Add Input**, pick that dataset. Settings: Accelerator **GPU T4 x2** (one is used),
   **Internet on**. Add-ons, Secrets: `MANO_USER`, `MANO_PASS` (your own mano.is.tue.mpg.de login), attached here.
4. Run all. Download `params.zip` from the Output tab, unzip it into `fit_inputs/` on your PC, then:
   `python fit_batch.py --unpack fit_inputs` and `python fit_mano.py sessions/<session> --post 1` for each session.

MANO stays inside this session (the licence doesn't allow sharing it): it is downloaded with your login and never
saved to the output.

In [ ]:
import glob, io, os, shutil, subprocess, sys, time, zipfile
import requests
from kaggle_secrets import UserSecretsClient

REPO = "/kaggle/working/repo"
WORK = "/kaggle/working/fit_inputs"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "https://github.com/amitabh-anandpd/multi-3di-synced-ir-capture", REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "smplx"], check=True)

sec = UserSecretsClient()
r = requests.post("https://download.is.tue.mpg.de/download.php?domain=mano&resume=1&sfile=mano_v1_2.zip",
                  data={"username": sec.get_secret("MANO_USER"), "password": sec.get_secret("MANO_PASS")}, timeout=900)
assert r.content[:2] == b"PK", f"MANO download failed (login / licence?): {r.text[:300]}"
os.makedirs(f"{REPO}/mano", exist_ok=True)
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    for side in ("LEFT", "RIGHT"):
        name = next(n for n in z.namelist() if n.endswith(f"models/MANO_{side}.pkl"))
        open(f"{REPO}/mano/MANO_{side}.pkl", "wb").write(z.read(name))
subprocess.run([sys.executable, f"{REPO}/tools/convert_mano.py"], check=True, capture_output=True)
print("MANO ready")

In [ ]:
inputs = sorted(p for p in glob.glob("/kaggle/input/**/*.npz", recursive=True) if not p.endswith("_params.npz"))
os.makedirs(WORK, exist_ok=True)
for p in inputs:
    shutil.copy(p, WORK)
print(len(inputs), "sessions:", [os.path.basename(p) for p in inputs])
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - turn on the GPU in Settings")

In [ ]:
t = time.time()
files = sorted(glob.glob(f"{WORK}/*.npz"))
files = [f for f in files if not f.endswith("_params.npz")]
subprocess.run([sys.executable, "-u", "fit_batch.py", "--fit", *files], cwd=REPO, check=True)
print(f"all fitted in {(time.time() - t) / 60:.1f} min")

In [ ]:
with zipfile.ZipFile("/kaggle/working/params.zip", "w") as z:
    for p in sorted(glob.glob(f"{WORK}/*_params.npz")):
        z.write(p, os.path.basename(p))
shutil.rmtree(f"{REPO}/mano")  # MANO must not end up in the saved output
print("download params.zip from the Output tab:", zipfile.ZipFile("/kaggle/working/params.zip").namelist())